# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassan9039/intern1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [11]:
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

My rule: I will prioritize content that shows signs of being a content-refresh opportunity. The score will give higher priority to pages with weaker search performance and signs that they may benefit from review. The rule is intended for decision-support, not as proof that a page definitely needs a refresh.

Reason codes:

LOW_CTR — the page receives relatively low clicks compared with its search impressions.
LOW_POSITION — the page has a relatively weak average search position.
LOW_VOLUME — the page has relatively low search impressions and may represent a lower-volume opportunity.

Action labels: REVIEW_REFRESH for high-priority pages and MONITOR for lower-priority pages.

In [12]:
ctr_check = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions = 0 THEN 'zero'
        WHEN gsc_clicks * 1.0 / gsc_impressions < 0.02 THEN 'low'
        WHEN gsc_clicks * 1.0 / gsc_impressions < 0.05 THEN 'medium'
        ELSE 'high'
    END AS ctr_bucket,
    COUNT(*) AS n
FROM {REL}
WHERE gsc_data_available IS TRUE
GROUP BY 1
ORDER BY 1
""").df()

ctr_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ctr_bucket,n
0,high,36490
1,low,3509198
2,medium,65373


CTR verdict: CONFIRMED. Most observed rows fall into the low-CTR bucket, so CTR provides a useful directional signal for identifying pages that may need review. This does not by itself prove that a page needs a refresh.

In [13]:
position_check = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position = 0 THEN 'no_data'
        WHEN gsc_avg_position <= 10 THEN 'top_10'
        WHEN gsc_avg_position <= 20 THEN '11_20'
        WHEN gsc_avg_position <= 50 THEN '21_50'
        ELSE '50_plus'
    END AS position_bucket,
    COUNT(*) AS n
FROM {REL}
WHERE gsc_data_available IS TRUE
GROUP BY 1
ORDER BY 1
""").df()

position_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n
0,11_20,519223
1,21_50,631491
2,50_plus,276863
3,no_data,163189
4,top_10,2020295


Average position verdict: CONFIRMED. A substantial number of rows fall outside the top 10, including 631,491 in positions 21–50 and 276,863 beyond position 50. This makes average position a useful directional signal for identifying weaker search visibility, although position alone does not prove that a page needs a refresh.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

My baseline rule: I rank content items using a simple score based on low CTR and weak average search position. Higher scores receive REVIEW_REFRESH, while lower scores receive MONITOR. The queue is ranked by score and then by impressions so that higher-volume opportunities are prioritized when scores are equal. The ranked queue is saved to work/outputs/baseline_action_score.csv.

In [14]:
import os
import pandas as pd
os.makedirs("work/outputs", exist_ok=True)

queue = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position
FROM {REL}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

queue["ctr"] = queue["gsc_clicks"] / queue["gsc_impressions"].replace(0, pd.NA)

queue["score"] = (
    ((queue["ctr"] < 0.02) * 2)
    + ((queue["gsc_avg_position"] > 20) * 1)
    + ((queue["gsc_avg_position"] > 50) * 1)
)

queue["reason_code"] = "LOW_VOLUME"

queue.loc[
    (queue["ctr"] < 0.02) & (queue["gsc_avg_position"] > 20),
    "reason_code"
] = "LOW_CTR"

queue.loc[
    (queue["ctr"] >= 0.02) & (queue["gsc_avg_position"] > 20),
    "reason_code"
] = "LOW_POSITION"

queue["action"] = queue["score"].apply(
    lambda x: "REVIEW_REFRESH" if x >= 2 else "MONITOR"
)

queue = queue.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action"
    ]
]

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows:", len(queue))
print("Saved: work/outputs/baseline_action_score.csv")

queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
Saved: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,score,reason_code,action
0,1,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,39405.0,3.0,0.000076,54.384283,4,LOW_CTR,REVIEW_REFRESH
1,2,client_23a62021009f63c4,content_2de9a39d3482a269,31664.0,6.0,0.000189,53.143951,4,LOW_CTR,REVIEW_REFRESH
2,3,client_23a62021009f63c4,content_1ff3c48911f11e70,27684.0,2.0,0.000072,55.147128,4,LOW_CTR,REVIEW_REFRESH
3,4,client_23a62021009f63c4,content_295e883e0e86ca3c,21939.0,0.0,0.000000,51.573254,4,LOW_CTR,REVIEW_REFRESH
4,5,client_23a62021009f63c4,content_0a9b787d28fc695c,21227.0,19.0,0.000895,52.293453,4,LOW_CTR,REVIEW_REFRESH
5,6,client_fef1a8f436438636,content_1c47c13983830602,20904.0,7.0,0.000335,56.597228,4,LOW_CTR,REVIEW_REFRESH
6,7,client_23a62021009f63c4,content_cbd0fdbc5c6a1ded,19298.0,13.0,0.000674,50.311703,4,LOW_CTR,REVIEW_REFRESH
7,8,client_23a62021009f63c4,content_736f0fed2392ab68,16179.0,1.0,0.000062,55.441210,4,LOW_CTR,REVIEW_REFRESH
8,9,client_20259bd6705d81d4,content_47da45b084a73115,14774.0,1.0,0.000068,50.756106,4,LOW_CTR,REVIEW_REFRESH
9,10,client_23a62021009f63c4,content_726afa8173b0a027,14248.0,6.0,0.000421,50.050285,4,LOW_CTR,REVIEW_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
import pandas as pd

queue = pd.read_csv("work/outputs/baseline_action_score.csv")
top20 = queue.head(20).copy()
top20

,rank,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,score,reason_code,action
0,1,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,39405.0,3.0,0.000076,54.384283,4,LOW_CTR,REVIEW_REFRESH
1,2,client_23a62021009f63c4,content_2de9a39d3482a269,31664.0,6.0,0.000189,53.143951,4,LOW_CTR,REVIEW_REFRESH
2,3,client_23a62021009f63c4,content_1ff3c48911f11e70,27684.0,2.0,0.000072,55.147128,4,LOW_CTR,REVIEW_REFRESH
3,4,client_23a62021009f63c4,content_295e883e0e86ca3c,21939.0,0.0,0.000000,51.573254,4,LOW_CTR,REVIEW_REFRESH
4,5,client_23a62021009f63c4,content_0a9b787d28fc695c,21227.0,19.0,0.000895,52.293453,4,LOW_CTR,REVIEW_REFRESH
5,6,client_fef1a8f436438636,content_1c47c13983830602,20904.0,7.0,0.000335,56.597228,4,LOW_CTR,REVIEW_REFRESH
6,7,client_23a62021009f63c4,content_cbd0fdbc5c6a1ded,19298.0,13.0,0.000674,50.311703,4,LOW_CTR,REVIEW_REFRESH
7,8,client_23a62021009f63c4,content_736f0fed2392ab68,16179.0,1.0,0.000062,55.441210,4,LOW_CTR,REVIEW_REFRESH
8,9,client_20259bd6705d81d4,content_47da45b084a73115,14774.0,1.0,0.000068,50.756106,4,LOW_CTR,REVIEW_REFRESH
9,10,client_23a62021009f63c4,content_726afa8173b0a027,14248.0,6.0,0.000421,50.050285,4,LOW_CTR,REVIEW_REFRESH


Top-20 review

1. Action: REVIEW_REFRESH — Why: Very low CTR with average position above 50. What could make it wrong: The page may target low-intent queries or have SERP features affecting clicks.

2. Action: REVIEW_REFRESH — Why: Very low CTR and weak average position. What could make it wrong: The page may not be intended to compete for high-volume search queries.

3. Action: REVIEW_REFRESH — Why: Extremely low CTR with average position above 55. What could make it wrong: Search intent or query mix may explain the low click-through rate.

4. Action: REVIEW_REFRESH — Why: Zero clicks despite substantial impressions and weak position. What could make it wrong: The impressions may come from queries where clicks are naturally unlikely.

5. Action: REVIEW_REFRESH — Why: Very low CTR despite more than 21,000 impressions and position above 50. What could make it wrong: The page may be appearing for poorly matched or low-intent queries.

6. Action: REVIEW_REFRESH — Why: Very low CTR with average position above 56. What could make it wrong: The page may have limited relevance to the queries generating impressions.

7. Action: REVIEW_REFRESH — Why: Low CTR combined with an average position above 50. What could make it wrong: Search demand or query intent may be the main cause rather than page quality.

8. Action: REVIEW_REFRESH — Why: Extremely low CTR with weak search visibility. What could make it wrong: The impressions may represent queries with little click potential.

9. Action: REVIEW_REFRESH — Why: Very low CTR and average position above 50. What could make it wrong: The page may be receiving impressions for irrelevant or low-intent queries.

10. Action: REVIEW_REFRESH — Why: Very low CTR with average position around 50. What could make it wrong: The low CTR may be expected because of the page's search position.

11. Action: REVIEW_REFRESH — Why: Low CTR despite nearly 14,000 impressions and weak position. What could make it wrong: Query intent may limit clicks regardless of content quality.

12. Action: REVIEW_REFRESH — Why: Very low CTR with average position above 50. What could make it wrong: The page may rank for queries that are not strong click opportunities.

13. Action: REVIEW_REFRESH — Why: Very low CTR and average position above 56. What could make it wrong: The search queries may not closely match the page's intended topic.

14. Action: REVIEW_REFRESH — Why: Very low CTR with position around 50. What could make it wrong: Low rankings alone may explain the limited clicks.

15. Action: REVIEW_REFRESH — Why: Very low CTR and average position above 50. What could make it wrong: The page may be exposed to low-intent queries.

16. Action: REVIEW_REFRESH — Why: Low CTR with more than 12,000 impressions and position around 50. What could make it wrong: The page may simply have limited visibility rather than a content-quality problem.

17. Action: REVIEW_REFRESH — Why: Very low CTR and average position near 60. What could make it wrong: Weak ranking may be the primary reason for low clicks.

18. Action: REVIEW_REFRESH — Why: Very low CTR with position above 51. What could make it wrong: Search intent may explain the low click rate.

19. Action: REVIEW_REFRESH — Why: Zero clicks despite nearly 12,000 impressions and weak position. What could make it wrong: The impressions may come from queries where users rarely click.

20. Action: REVIEW_REFRESH — Why: Zero clicks with more than 11,000 impressions and position above 50. What could make it wrong: Low ranking or query intent may explain the absence of clicks.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks + leakage check

The weakest picks are the rows where the recommendation may be driven mainly by low search visibility rather than a true content-quality issue. In particular, pages with very low CTR and average position above 50 should be treated as review candidates rather than confirmed refresh opportunities.

The baseline uses only March 2026 search performance fields: impressions, clicks, CTR, and average position. No future-window metrics or label-derived features are used. Product flags are not used as scoring inputs.

In [16]:
import pandas as pd

required_columns = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "score",
    "reason_code",
    "action"
}

future_or_label_columns = {
    "label",
    "target",
    "future_clicks",
    "future_impressions",
    "future_ctr",
    "leaked_label"
}

used_columns = set(queue.columns)

print("Required scoring columns present:", required_columns.issubset(used_columns))
print("Future or label-derived columns present:", bool(used_columns & future_or_label_columns))
print("Leakage check:", "PASS" if not (used_columns & future_or_label_columns) else "FAIL")

Required scoring columns present: True
Future or label-derived columns present: False
Leakage check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.